In [ ]:
import dask
import time
import toolviper
import random

import toolviper.utils.logger as logger
import toolviper.utils.display as display

import toolviper.dask.client as client

from collections import defaultdict

In [ ]:
client = client.local_client(
    cores=10,
    log_params={
        "log_to_file":False,
        "log_to_term":True,
        "log_level":"DEBUG" 
    },
    worker_log_params={
        "log_to_file":False,
        "log_to_term":True,
        "log_level":"DEBUG" 
    }
)

# Spawn dashboard window in a seperate tab,
# comment out if you don't want this to spawn.
# webbrowser.open(url=client.dashboard_link)

In [ ]:
data = toolviper.utils.sd.prototype.simulate(field=1, spw=2, polarization=["XX", "YY"], antenna=1, row=2)
data

In [ ]:
def gather(result):
    if result is None:
        return
        
    return result

# Simple function to generate a time delay and simulate 
# data processing
def generate_delay(n=1, m=2):
    time.sleep(random.uniform(n, m))

In [ ]:
def imaging_parameter_setup(*arg, **kwargs):
    generate_delay(n=1, m=3)

In [ ]:
class Graph:
    def __init__(self):
        self._graph = None
        self._results = defaultdict(list)
        
    def source(self, function, axes, connect=False):
        function_name = function.__name__
        previous = None
        
        if connect:
            previous = self._graph
            
        self._graph = toolviper.utils.sd.distribute(
                dataset=data,
                axes=axes,
                function=function,
                previous=previous
        )
        
        self._results[function_name].append(self._graph)

    def sink(self, function, edges=None):
        self._graph = dask.delayed(function)(self._graph)

    def visualize(self):
        return dask.visualize(self._graph)

    def compute(self):
        return dask.compute(self._graph)

In [ ]:
def check_values(*args, **kwargs):
    print(f"check::args::{args}\n")

def set_values(*args, **kwargs):
    print(f"set::args::{args}\n")

In [ ]:
#dask.visualize(new)

In [ ]:
#dask.compute(new)

job = {
    "dataset": data,
    "function": check_values
}

In [ ]:
graph = toolviper.utils.sd.graph.Graph()

graph.source(job=job, axes=["field", "antenna"])
job["function"] = set_values

graph.source(job=job, axes=["field", "spw", "polarization"], connect=True, node="check_values")
graph.sink(function=gather)

In [ ]:
graph.visualize()

In [ ]:
graph.compute()

In [ ]:
graph.nodes